In [5]:
# =========================
# ICDM TSF benchmark downloader - FIXED VERSION
# =========================

import os
import re
import zipfile
import shutil
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import time

ROOT = Path("data_icdm_tsf")
RAW = ROOT / "_raw"
LONG_RAW = RAW / "long_horizon"
M4_DIR = ROOT / "M4"

ROOT.mkdir(exist_ok=True)
RAW.mkdir(exist_ok=True)
LONG_RAW.mkdir(parents=True, exist_ok=True)
M4_DIR.mkdir(parents=True, exist_ok=True)

# Standard long-horizon datasets
LONG_HORIZON_ZIP_URL = "https://www.dropbox.com/s/rlc1qmprpvuqrsv/all_six_datasets.zip?dl=1"

# Official M4 repository files
M4_BASE = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"
M4_FILES = [
    "M4-info.csv",
    "Train/Hourly-train.csv",
    "Train/Daily-train.csv",
    "Train/Weekly-train.csv",
    "Train/Monthly-train.csv",
    "Train/Quarterly-train.csv",
    "Train/Yearly-train.csv",
    "Test/Hourly-test.csv",
    "Test/Daily-test.csv",
    "Test/Weekly-test.csv",
    "Test/Monthly-test.csv",
    "Test/Quarterly-test.csv",
    "Test/Yearly-test.csv",
]


def download_file(url: str, out_path: Path, chunk_size: int = 1024 * 1024, timeout: int = 120, retries: int = 3):
    """Download with retry logic and longer timeout"""
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[skip] {out_path}")
        return

    for attempt in range(retries):
        try:
            print(f"[download] {url} (attempt {attempt+1}/{retries})")
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))

                with open(out_path, "wb") as f, tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    desc=out_path.name
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            return  # Success
        except requests.exceptions.RequestException as e:
            print(f"[warn] Attempt {attempt+1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                print(f"[error] Failed to download {url} after {retries} attempts")
                raise


def unzip_file(zip_path: Path, out_dir: Path):
    print(f"[unzip] {zip_path} -> {out_dir}")
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        # Filter out __MACOSX metadata folders
        for member in zf.namelist():
            if not member.startswith('__MACOSX/'):
                zf.extract(member, out_dir)


# =========================
# 1) Download long-horizon benchmark zip
# =========================

zip_path = RAW / "all_six_datasets.zip"
download_file(LONG_HORIZON_ZIP_URL, zip_path)

# unzip only once
marker = LONG_RAW / ".unzipped"
if not marker.exists():
    unzip_file(zip_path, LONG_RAW)
    marker.write_text("done", encoding="utf-8")
else:
    print("[skip] long-horizon archive already unzipped")


# =========================
# 2) Collect CSV files from extracted archive
# =========================

# Find actual CSV files, excluding __MACOSX metadata
csv_files = [p for p in LONG_RAW.rglob("*.csv") if "__MACOSX" not in str(p)]
print(f"\nFound {len(csv_files)} CSV files in long-horizon archive:")
for p in csv_files:
    print(" -", p)

# Map dataset names to their subdirectory patterns in the archive
# The archive structure is: all_six_datasets/{DATASET_NAME}/Y_df.csv
TARGET_MAPPING = {
    "ETTh1": "all_six_datasets/ETTh1/Y_df.csv",
    "ETTh2": "all_six_datasets/ETTh2/Y_df.csv",
    "ETTm1": "all_six_datasets/ETTm1/Y_df.csv",
    "ETTm2": "all_six_datasets/ETTm2/Y_df.csv",
    "Weather": "all_six_datasets/Weather/Y_df.csv",
    "Traffic": "all_six_datasets/TrafficL/Y_df.csv",  # Note: TrafficL in archive
    "Electricity": "all_six_datasets/ECL/Y_df.csv",    # Note: ECL in archive
    "Exchange": "all_six_datasets/Exchange/Y_df.csv",
    "ILI": "all_six_datasets/ILI/Y_df.csv",
}

copied = {}

for dataset_name, relative_path in TARGET_MAPPING.items():
    # Search for the file in extracted contents
    matches = [p for p in csv_files if relative_path.replace('\\', '/') in str(p).replace('\\', '/')]
    
    if not matches:
        # Fallback: try to find by dataset name in path
        matches = [p for p in csv_files if dataset_name.lower() in str(p).lower() and p.name == "Y_df.csv"]
    
    if not matches:
        print(f"[warn] not found: {dataset_name}")
        continue

    src = matches[0]
    dst_dir = ROOT / dataset_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"{dataset_name}.csv"

    shutil.copy2(src, dst)
    copied[dataset_name] = dst
    print(f"[copy] {dataset_name}: {src.name} -> {dst}")


# =========================
# 3) Download M4 with retry logic
# =========================

print("\n=== Downloading M4 datasets ===")
for rel in M4_FILES:
    url = f"{M4_BASE}/{rel}"
    out = M4_DIR / rel
    try:
        download_file(url, out, timeout=120, retries=3)
    except Exception as e:
        print(f"[error] Could not download {rel}: {e}")
        print("[info] You may need to download M4 files manually from: https://github.com/Mcompetitions/M4-methods/tree/master/Dataset")


# =========================
# 4) Quick integrity check
# =========================

summary = []

for name, path in copied.items():
    if path.exists():
        try:
            df = pd.read_csv(path)
            summary.append({
                "dataset": name,
                "file": str(path),
                "rows": len(df),
                "columns": len(df.columns),
                "column_names_head": ", ".join(map(str, df.columns[:8]))
            })
        except Exception as e:
            summary.append({
                "dataset": name,
                "file": str(path),
                "rows": "ERROR",
                "columns": "ERROR",
                "column_names_head": str(e)
            })

# M4 check
for freq in ["Hourly", "Daily", "Weekly", "Monthly", "Quarterly", "Yearly"]:
    train_path = M4_DIR / "Train" / f"{freq}-train.csv"
    test_path = M4_DIR / "Test" / f"{freq}-test.csv"

    for p, suffix in [(train_path, "train"), (test_path, "test")]:
        if p.exists():
            try:
                df = pd.read_csv(p)
                summary.append({
                    "dataset": f"M4-{freq}-{suffix}",
                    "file": str(p),
                    "rows": len(df),
                    "columns": len(df.columns),
                    "column_names_head": ", ".join(map(str, df.columns[:8]))
                })
            except Exception as e:
                summary.append({
                    "dataset": f"M4-{freq}-{suffix}",
                    "file": str(p),
                    "rows": "ERROR",
                    "columns": "ERROR",
                    "column_names_head": str(e)
                })

if summary:
    summary_df = pd.DataFrame(summary)
    print("\n=== Dataset Summary ===")
    display(summary_df)

print(f"\n✅ Done. Main data folder: {ROOT.resolve()}")

[skip] data_icdm_tsf/_raw/all_six_datasets.zip
[skip] long-horizon archive already unzipped

Found 9 CSV files in long-horizon archive:
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/Weather/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/ETTm2/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/ETTh2/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/ETTh1/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/TrafficL/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/ILI/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/ETTm1/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/ECL/Y_df.csv
 - data_icdm_tsf/_raw/long_horizon/all_six_datasets/Exchange/Y_df.csv
[copy] ETTh1: Y_df.csv -> data_icdm_tsf/ETTh1/ETTh1.csv
[copy] ETTh2: Y_df.csv -> data_icdm_tsf/ETTh2/ETTh2.csv
[copy] ETTm1: Y_df.csv -> data_icdm_tsf/ETTm1/ETTm1.csv
[copy] ETTm2: Y_df.csv -> data_icdm_tsf/ETTm2/ETTm2.csv
[copy] Weather: Y_df.csv -> data_

Quarterly-test.csv:   0%|          | 0.00/734k [00:00<?, ?B/s]

[download] https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Test/Yearly-test.csv (attempt 1/3)


Yearly-test.csv:   0%|          | 0.00/563k [00:00<?, ?B/s]


=== Dataset Summary ===


,dataset,file,rows,columns,column_names_head
0,ETTh1,data_icdm_tsf/ETTh1/ETTh1.csv,17420,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT"
1,ETTh2,data_icdm_tsf/ETTh2/ETTh2.csv,17420,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT"
2,ETTm1,data_icdm_tsf/ETTm1/ETTm1.csv,69680,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT"
3,ETTm2,data_icdm_tsf/ETTm2/ETTm2.csv,69680,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT"
4,Weather,data_icdm_tsf/Weather/Weather.csv,52695,22,"date, p (mbar), T (degC), Tpot (K), Tdew (degC..."
5,Traffic,data_icdm_tsf/Traffic/Traffic.csv,17544,863,"date, 0, 1, 2, 3, 4, 5, 6"
6,Electricity,data_icdm_tsf/Electricity/Electricity.csv,26304,322,"date, 0, 1, 2, 3, 4, 5, 6"
7,Exchange,data_icdm_tsf/Exchange/Exchange.csv,7588,9,"date, 0, 1, 2, 3, 4, 5, 6"
8,ILI,data_icdm_tsf/ILI/ILI.csv,966,8,"date, % WEIGHTED ILI, %UNWEIGHTED ILI, AGE 0-4..."
9,M4-Hourly-train,data_icdm_tsf/M4/Train/Hourly-train.csv,414,961,"V1, V2, V3, V4, V5, V6, V7, V8"



✅ Done. Main data folder: /home/tahiti/LLM/data_icdm_tsf


In [2]:
# ============================================================
# FIX: Download M4 from official M4 GitHub repository
# Run this AFTER the previous Hugging Face downloader failed on M4
# ============================================================

import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("data_icdm_tsf")
M4_DIR = ROOT / "M4"
M4_DIR.mkdir(parents=True, exist_ok=True)

M4_BASE_URL = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset"

M4_FILES = [
    "M4-info.csv",
    "Train/Hourly-train.csv",
    "Train/Daily-train.csv",
    "Train/Weekly-train.csv",
    "Train/Monthly-train.csv",
    "Train/Quarterly-train.csv",
    "Train/Yearly-train.csv",
    "Test/Hourly-test.csv",
    "Test/Daily-test.csv",
    "Test/Weekly-test.csv",
    "Test/Monthly-test.csv",
    "Test/Quarterly-test.csv",
    "Test/Yearly-test.csv",
]

def download_url(url: str, out_path: Path, chunk_size: int = 1024 * 1024):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[skip] {out_path}")
        return

    print(f"[download] {url}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(out_path, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=out_path.name
        ) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    print(f"[ok] {out_path} | {out_path.stat().st_size / 1024 / 1024:.2f} MB")

for rel_path in M4_FILES:
    url = f"{M4_BASE_URL}/{rel_path}"
    out_path = M4_DIR / rel_path
    download_url(url, out_path)

print("\nM4 download complete.")

[skip] data_icdm_tsf/M4/M4-info.csv
[skip] data_icdm_tsf/M4/Train/Hourly-train.csv
[download] https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Train/Daily-train.csv


Daily-train.csv:   0%|          | 0.00/32.5M [00:00<?, ?B/s]

[ok] data_icdm_tsf/M4/Train/Daily-train.csv | 91.33 MB
[download] https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Train/Weekly-train.csv


Weekly-train.csv:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

[ok] data_icdm_tsf/M4/Train/Weekly-train.csv | 3.83 MB
[download] https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Train/Monthly-train.csv


...

Weekly-test.csv:   0%|          | 0.00/17.1k [00:00<?, ?B/s]

[ok] data_icdm_tsf/M4/Test/Weekly-test.csv | 0.04 MB
[download] https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Test/Monthly-test.csv


Monthly-test.csv:   0%|          | 0.00/2.70M [00:00<?, ?B/s]

[ok] data_icdm_tsf/M4/Test/Monthly-test.csv | 7.57 MB
[download] https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Test/Quarterly-test.csv


ReadTimeout: HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Read timed out. (read timeout=120)

In [6]:
from pathlib import Path
import pandas as pd
import os

ROOT = Path("data_icdm_tsf")

EXPECTED_FILES = {
    "ETTh1": ROOT / "ETTh1" / "ETTh1.csv",
    "ETTh2": ROOT / "ETTh2" / "ETTh2.csv",
    "ETTm1": ROOT / "ETTm1" / "ETTm1.csv",
    "ETTm2": ROOT / "ETTm2" / "ETTm2.csv",
    "Weather": ROOT / "Weather" / "Weather.csv",
    "Traffic": ROOT / "Traffic" / "Traffic.csv",
    "Electricity": ROOT / "Electricity" / "Electricity.csv",
    "Exchange": ROOT / "Exchange" / "Exchange.csv",
    "ILI": ROOT / "ILI" / "ILI.csv",

    "M4-info": ROOT / "M4" / "M4-info.csv",

    "M4-Hourly-train": ROOT / "M4" / "Train" / "Hourly-train.csv",
    "M4-Daily-train": ROOT / "M4" / "Train" / "Daily-train.csv",
    "M4-Weekly-train": ROOT / "M4" / "Train" / "Weekly-train.csv",
    "M4-Monthly-train": ROOT / "M4" / "Train" / "Monthly-train.csv",
    "M4-Quarterly-train": ROOT / "M4" / "Train" / "Quarterly-train.csv",
    "M4-Yearly-train": ROOT / "M4" / "Train" / "Yearly-train.csv",

    "M4-Hourly-test": ROOT / "M4" / "Test" / "Hourly-test.csv",
    "M4-Daily-test": ROOT / "M4" / "Test" / "Daily-test.csv",
    "M4-Weekly-test": ROOT / "M4" / "Test" / "Weekly-test.csv",
    "M4-Monthly-test": ROOT / "M4" / "Test" / "Monthly-test.csv",
    "M4-Quarterly-test": ROOT / "M4" / "Test" / "Quarterly-test.csv",
    "M4-Yearly-test": ROOT / "M4" / "Test" / "Yearly-test.csv",
}


def check_csv(name, path):
    result = {
        "dataset": name,
        "path": str(path),
        "exists": False,
        "size_mb": None,
        "readable": False,
        "rows": None,
        "cols": None,
        "columns_head": None,
        "na_total": None,
        "status": "PROBLEM",
        "error": "",
    }

    if not path.exists():
        result["error"] = "file not found"
        return result

    result["exists"] = True
    result["size_mb"] = round(path.stat().st_size / 1024 / 1024, 3)

    if path.stat().st_size == 0:
        result["error"] = "empty file"
        return result

    try:
        df = pd.read_csv(path)
        result["readable"] = True
        result["rows"] = len(df)
        result["cols"] = len(df.columns)
        result["columns_head"] = ", ".join(map(str, df.columns[:10]))
        result["na_total"] = int(df.isna().sum().sum())

        if len(df) == 0:
            result["error"] = "zero rows"
            return result

        if len(df.columns) == 0:
            result["error"] = "zero columns"
            return result

        result["status"] = "OK"
        return result

    except Exception as e:
        result["error"] = repr(e)
        return result


results = []

print("=" * 100)
print("CHECKING DATASETS")
print("=" * 100)

for name, path in EXPECTED_FILES.items():
    r = check_csv(name, path)
    results.append(r)

    mark = "✅" if r["status"] == "OK" else "❌"
    print(f"{mark} {name:22s} | {r['status']:8s} | {path}")

    if r["status"] != "OK":
        print(f"   ERROR: {r['error']}")
    else:
        print(f"   shape=({r['rows']}, {r['cols']}), size={r['size_mb']} MB")
        print(f"   columns: {r['columns_head']}")

summary_df = pd.DataFrame(results)

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

display(summary_df)

print("\n" + "=" * 100)
print("PROBLEMS ONLY")
print("=" * 100)

problems = summary_df[summary_df["status"] != "OK"]

if len(problems) == 0:
    print("✅ ALL EXPECTED FILES ARE PRESENT AND READABLE")
else:
    display(problems)
    print(f"❌ Problems found: {len(problems)} / {len(summary_df)}")


print("\n" + "=" * 100)
print("ALL FILES FOUND UNDER data_icdm_tsf")
print("=" * 100)

all_files = sorted([p for p in ROOT.rglob("*") if p.is_file()])

for p in all_files:
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"{p} | {size_mb:.3f} MB")

CHECKING DATASETS
✅ ETTh1                  | OK       | data_icdm_tsf/ETTh1/ETTh1.csv
   shape=(17420, 8), size=2.47 MB
   columns: date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT
✅ ETTh2                  | OK       | data_icdm_tsf/ETTh2/ETTh2.csv
   shape=(17420, 8), size=2.306 MB
   columns: date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT
✅ ETTm1                  | OK       | data_icdm_tsf/ETTm1/ETTm1.csv
   shape=(69680, 8), size=9.881 MB
   columns: date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT
✅ ETTm2                  | OK       | data_icdm_tsf/ETTm2/ETTm2.csv
   shape=(69680, 8), size=9.229 MB
   columns: date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT
✅ Weather                | OK       | data_icdm_tsf/Weather/Weather.csv
   shape=(52695, 22), size=6.9 MB
   columns: date, p (mbar), T (degC), Tpot (K), Tdew (degC), rh (%), VPmax (mbar), VPact (mbar), VPdef (mbar), sh (g/kg)
✅ Traffic                | OK       | data_icdm_tsf/Traffic/Traffic.csv
   shape=(17544, 863), size=130.156 MB
   co

,dataset,path,exists,size_mb,readable,rows,cols,columns_head,na_total,status,error
0,ETTh1,data_icdm_tsf/ETTh1/ETTh1.csv,True,2.470,True,17420,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT",0,OK,
1,ETTh2,data_icdm_tsf/ETTh2/ETTh2.csv,True,2.306,True,17420,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT",0,OK,
2,ETTm1,data_icdm_tsf/ETTm1/ETTm1.csv,True,9.881,True,69680,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT",0,OK,
3,ETTm2,data_icdm_tsf/ETTm2/ETTm2.csv,True,9.229,True,69680,8,"date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT",0,OK,
4,Weather,data_icdm_tsf/Weather/Weather.csv,True,6.900,True,52695,22,"date, p (mbar), T (degC), Tpot (K), Tdew (degC...",0,OK,
5,Traffic,data_icdm_tsf/Traffic/Traffic.csv,True,130.156,True,17544,863,"date, 0, 1, 2, 3, 4, 5, 6, 7, 8",0,OK,
6,Electricity,data_icdm_tsf/Electricity/Electricity.csv,True,91.154,True,26304,322,"date, 0, 1, 2, 3, 4, 5, 6, 7, 8",0,OK,
7,Exchange,data_icdm_tsf/Exchange/Exchange.csv,True,0.608,True,7588,9,"date, 0, 1, 2, 3, 4, 5, 6, OT",0,OK,
8,ILI,data_icdm_tsf/ILI/ILI.csv,True,0.064,True,966,8,"date, % WEIGHTED ILI, %UNWEIGHTED ILI, AGE 0-4...",0,OK,
9,M4-info,data_icdm_tsf/M4/M4-info.csv,True,4.135,True,100000,6,"M4id, category, Frequency, Horizon, SP, Starti...",0,OK,



PROBLEMS ONLY
✅ ALL EXPECTED FILES ARE PRESENT AND READABLE

ALL FILES FOUND UNDER data_icdm_tsf
data_icdm_tsf/ETTh1/ETTh1.csv | 2.470 MB
data_icdm_tsf/ETTh2/ETTh2.csv | 2.306 MB
data_icdm_tsf/ETTm1/ETTm1.csv | 9.881 MB
data_icdm_tsf/ETTm2/ETTm2.csv | 9.229 MB
data_icdm_tsf/Electricity/Electricity.csv | 91.154 MB
data_icdm_tsf/Exchange/Exchange.csv | 0.608 MB
data_icdm_tsf/ILI/ILI.csv | 0.064 MB
data_icdm_tsf/M4/M4-info.csv | 4.135 MB
data_icdm_tsf/M4/Test/Daily-test.csv | 0.550 MB
data_icdm_tsf/M4/Test/Hourly-test.csv | 0.127 MB
data_icdm_tsf/M4/Test/Monthly-test.csv | 7.575 MB
data_icdm_tsf/M4/Test/Quarterly-test.csv | 1.880 MB
data_icdm_tsf/M4/Test/Weekly-test.csv | 0.042 MB
data_icdm_tsf/M4/Test/Yearly-test.csv | 1.418 MB
data_icdm_tsf/M4/Train/Daily-train.csv | 91.329 MB
data_icdm_tsf/M4/Train/Hourly-train.csv | 2.238 MB
data_icdm_tsf/M4/Train/Monthly-train.csv | 87.409 MB
data_icdm_tsf/M4/Train/Quarterly-train.csv | 36.992 MB
data_icdm_tsf/M4/Train/Weekly-train.csv | 3.829 MB
da

In [7]:
!pip install -U transformers huggingface_hub accelerate safetensors sentencepiece

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai